In [ ]:
import albumentations as A
import cv2 as cv
import matplotlib.pyplot as plt
import numpy as np
import os
import segmentation_models as sm
import tensorflow as tf
import tensorflow_addons as tfa
import tensorflow.keras.backend as K
import wandb
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.optimizers import Adam
from wandb.keras import WandbCallback

from unet import *

print(f'TensorFlow version: {tf.__version__}')
print(f'GPUs available: {tf.config.list_physical_devices("GPU")}')

In [ ]:
# Segmentation classes
CLASS_NAMES = [
    'Background (waterbody)', 'Human divers', 'Aquatic plants and sea-grass',
    'Wrecks and ruins', 'Robots (AUVs/ROVs/instruments)', 'Reefs and invertebrates',
    'Fish and vertebrates', 'Sea-floor and rocks',
]
NUM_CLASSES = len(CLASS_NAMES)
BATCH_SIZE = 4
EPOCHS = 50  # 50
STEPS_PER_EPOCH = 1500  # 5000
LEARNING_RATE = 3e-4
MOMENTUM = 0.5
DROPOUT = 0.0

HEIGHT = 240  # 480
WIDTH = 320  # 640
CHANNELS = 3
IMAGE_SHAPE = (HEIGHT, WIDTH, CHANNELS)

DEEP_SUPERVISION = False
LOGGING = False

CLASS_TO_COLOR = {
    0: (0, 0, 0),  # Black
    1: (0, 0, 1),  # Blue
    2: (0, 1, 0),  # Green
    3: (0, 1, 1),  # Sky
    4: (1, 0, 0),  # Red
    5: (1, 0, 1),  # Pink
    6: (1, 1, 0),  # Yellow
    7: (1, 1, 1),  # White
}
COLOR_TO_CLASS = {v: k for k, v in CLASS_TO_COLOR.items()}
CLASS_COLORS = np.array([list(color) for color in CLASS_TO_COLOR.values()])
CLASS_WEIGHTS = np.array([1.06, 14.62, 15.72, 5.10, 44.18, 1.00, 4.78, 2.42])

config = {
    'model': 'UNet',
    'batch_size': BATCH_SIZE,
    'epochs': EPOCHS,
    'steps_per_epoch': STEPS_PER_EPOCH,
    'loss': 'categorical_crossentropy',
    'optimizer': 'Adam',
    'learning_rate': LEARNING_RATE,
    'momentum': MOMENTUM,
    'dropout': DROPOUT,
    'image_shape': IMAGE_SHAPE,
    'deep_supervision': DEEP_SUPERVISION,
}

In [ ]:
if LOGGING:
    wandb.login()
    wandb.init(project='PVID-assignment-4', config=config)

## Preprocessing

In [ ]:
def list_files(path):
    content = os.listdir(path)
    # Get only files (not directories)
    files = [os.path.join(path, f) for f in content if os.path.isfile(os.path.join(path, f))]
    # Ensure the lists are sorted to match the corresponding image and mask
    files.sort()
    return files


def load_image(path):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=CHANNELS)
    image = tf.image.convert_image_dtype(image, tf.float32)
    image = tf.image.resize(image, (HEIGHT, WIDTH), method='bilinear')
    return image


def load_mask(path):
    mask = tf.io.read_file(path)
    mask = tf.image.decode_bmp(mask, channels=CHANNELS)
    mask = tf.where(mask > 127, 1, 0)
    mask = tf.image.resize(mask, (HEIGHT, WIDTH), method='nearest')
    mask = tf.cast(mask, tf.float32)
    return mask


# Encode RGB mask to class indices in range [0, num_classes - 1]
def rgb_to_class_indices(mask):
    class_colors_tensor = tf.constant(CLASS_COLORS, dtype=tf.float32)
    # Calculate the similarity between the RGB mask and class colors
    similarity = tf.reduce_sum(tf.abs(tf.subtract(mask, class_colors_tensor[:, None, None, :])), axis=-1)
    # Find the index of the class color with the minimum similarity
    class_indices_mask = tf.argmin(similarity, axis=0)
    return class_indices_mask


# One-hot encode class indices mask (e.g. [1, 3, 0] -> [[0, 1, 0, 0], [0, 0, 0, 1], [1, 0, 0, 0]])
def class_indices_to_one_hot(mask):
    one_hot_mask = tf.one_hot(mask, NUM_CLASSES, dtype=tf.int32)
    one_hot_mask = tf.cast(one_hot_mask, tf.float32)
    return one_hot_mask


# Encode RGB mask to one-hot mask (e.g. [[0, 0, 1], [1, 0, 0]] -> [[0, 1, 0, 0], [1, 0, 0, 0]])
def rgb_to_one_hot(mask):
    indices = rgb_to_class_indices(mask)
    one_hot_mask = class_indices_to_one_hot(indices)
    return one_hot_mask


# Decode one-hot mask to RGB mask
def class_indices_to_rgb(mask):
    class_colors_tensor = tf.constant(CLASS_COLORS, dtype=tf.float32)
    # Get the RGB color for each pixel from the class color
    rgb_mask = tf.gather(class_colors_tensor, tf.cast(mask, tf.int32))
    return rgb_mask


def one_hot_to_class_indices(mask):
    indices = tf.argmax(mask, axis=-1)
    return indices


def one_hot_to_rgb(mask):
    indices = one_hot_to_class_indices(mask)
    rgb_mask = class_indices_to_rgb(indices)
    return rgb_mask


def visualize_pair(image, mask, mask_type='rgb'):
    if mask_type == 'rgb':
        pass
    elif mask_type == 'one-hot':
        mask = one_hot_to_rgb(mask)
    elif mask_type == 'class-indices':
        mask = class_indices_to_rgb(mask)
    else:
        raise ValueError(f'Unknown mask type: {mask_type}')
    mask = mask.numpy().astype(np.uint8)
    mask *= 255

    plt.figure(figsize=(10, 5))
    plt.subplot(121)
    plt.imshow(image)
    plt.axis('off')
    plt.subplot(122)
    plt.imshow(mask)
    plt.axis('off')
    plt.show()


# Augment with tensorflow
def augmentation(image, mask):
    # Randomly flip the image horizontally
    if tf.random.uniform(()) > 0.5:
        image = tf.image.flip_left_right(image)
        mask = tf.image.flip_left_right(mask)

    # Randomly rotate the image by random angle
    if tf.random.uniform(()) > 0.5:
        degrees = 50
        radian = degrees * np.pi / 180
        angle = tf.random.uniform(shape=[], minval=-radian, maxval=radian)
        image = tfa.image.rotate(image, tf.cast(angle, tf.float32), interpolation='NEAREST', fill_mode='reflect')
        mask = tfa.image.rotate(mask, tf.cast(angle, tf.float32), interpolation='NEAREST', fill_mode='reflect')

    # Randomly adjust brightness
    if tf.random.uniform(()) > 0.6:
        delta = tf.random.uniform(shape=[], minval=-0.1, maxval=0.1)
        image = tf.image.adjust_brightness(image, delta=delta)
        image = tf.clip_by_value(image, 0, 1)

    # Randomly blur the image (mean or gaussian)
    if tf.random.uniform(()) > 0.6:
        if tf.random.uniform(()) > 0.5:
            image = tfa.image.mean_filter2d(image, filter_shape=3)
        else:
            image = tfa.image.gaussian_filter2d(image, filter_shape=3)

    return image, mask


def load_and_preprocess(image_path, mask_path):
    image = load_image(image_path)
    mask = load_mask(mask_path)
    return image, mask


def load_dataset(path, val_split=0.2, seed=411, encode='one-hot', repeat=True, augment=True):
    # Get file names
    train_val_images = list_files(os.path.join(path, 'train_val/images'))
    train_val_masks = list_files(os.path.join(path, 'train_val/masks'))
    test_images = list_files(os.path.join(path, 'TEST/images'))
    test_masks = list_files(os.path.join(path, 'TEST/masks'))

    # Split to train and validation
    train_images, val_images, train_masks, val_masks = train_test_split(
        train_val_images, train_val_masks, test_size=val_split, random_state=seed
    )

    # Create datasets
    train_dataset = tf.data.Dataset.from_tensor_slices((train_images, train_masks))
    val_dataset = tf.data.Dataset.from_tensor_slices((val_images, val_masks))
    test_dataset = tf.data.Dataset.from_tensor_slices((test_images, test_masks))

    # Preprocess datasets
    train_dataset = train_dataset.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    val_dataset = val_dataset.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    test_dataset = test_dataset.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)

    # Cache datasets
    train_dataset = train_dataset.cache()
    val_dataset = val_dataset.cache()
    test_dataset = test_dataset.cache()

    # Augment datasets
    if augment:
        train_dataset = train_dataset.map(augmentation, num_parallel_calls=tf.data.AUTOTUNE)

    # Encode masks (rgb, one-hot or class indices)
    if encode == 'rgb':
        encode_fn = lambda x, y: (x, y)
    elif encode == 'one-hot':
        encode_fn = lambda x, y: (x, rgb_to_one_hot(y))
    elif encode == 'class-indices':
        encode_fn = lambda x, y: (x, rgb_to_class_indices(y))
    else:
        raise ValueError(f'Unknown encode type: {encode}')

    train_dataset = train_dataset.map(encode_fn, num_parallel_calls=tf.data.AUTOTUNE)
    val_dataset = val_dataset.map(encode_fn, num_parallel_calls=tf.data.AUTOTUNE)
    test_dataset = test_dataset.map(encode_fn, num_parallel_calls=tf.data.AUTOTUNE)

    # Shuffle and batch datasets
    train_dataset = train_dataset.shuffle(len(train_images)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    val_dataset = val_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    test_dataset = test_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

    if repeat:
        train_dataset = train_dataset.repeat()

    # Return datasets
    return train_dataset, val_dataset, test_dataset


In [ ]:
i = 0
image, mask = load_image(list_files('input/train_val/images')[i]), load_mask(list_files('input/train_val/masks')[i])
image.shape, mask.shape

In [ ]:
# Load dataset
dataset_path = 'input/'
train_ds, val_ds, test_ds = load_dataset(dataset_path, val_split=0.1)

In [ ]:
for code, name, weight in zip(CLASS_TO_COLOR.keys(), CLASS_NAMES, CLASS_WEIGHTS):
    print(f'Class {code}: {name} [weight = {weight}]')
print()

try:
    print('Number of batches in training set:', len(train_ds))
except TypeError:
    print('Training set is repeated indefinitely.')
print('Number of batches in validation set:', len(val_ds))
print('Number of batches in test set:', len(test_ds))
print()

data_batch = next(iter(val_ds))
images, masks = data_batch
print('Image batch shape:', images.shape)
print('Mask batch shape:', masks.shape)

reshaped = np.reshape(masks, (-1, 3))
mask_pixels = np.unique(reshaped, axis=0)
print('Unique mask pixels:\n', mask_pixels)

# Show some iamge-mask pairs for sanity check
for i, (img, mask) in enumerate(zip(images, masks)):
    if i > 4: break
    visualize_pair(img, mask, mask_type='one-hot')

## Training

In [ ]:
def iou_index(y_true, y_pred, smooth=1):
    intersection = K.sum(K.abs(y_true * y_pred), axis=[1, 2, 3])
    union = K.sum(y_true, axis=[1, 2, 3]) + K.sum(y_pred, axis=[1, 2, 3]) - intersection
    iou = K.mean((intersection + smooth) / (union + smooth), axis=0)
    return iou


def f_score(y_true, y_pred, smooth=1):
    intersection = K.sum(y_true * y_pred, axis=[1, 2, 3])
    union = K.sum(y_true, axis=[1, 2, 3]) + K.sum(y_pred, axis=[1, 2, 3])
    dice = K.mean((2. * intersection + smooth) / (union + smooth), axis=0)
    return dice


# IoU = Jaccard index
def jaccard_coef(y_true, y_pred, smooth: float = 1.0):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(K.abs(y_true_f * y_pred_f))
    union = K.sum(y_true_f) + K.sum(y_pred_f) - intersection
    return K.mean((intersection + smooth) / (union + smooth))


# f1 score = Dice coefficient
def dice_coef(y_true, y_pred, smooth: float = 1.0):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    area_true = K.sum(y_true_f)
    area_pred = K.sum(y_pred_f)
    return K.mean((2. * intersection + smooth) / (area_true + area_pred + smooth))


def jaccard_loss(y_true, y_pred):
    return 1.0 - jaccard_coef(y_true, y_pred)


def dice_loss(y_true, y_pred):
    return 1.0 - dice_coef(y_true, y_pred)


def focal_loss(y_true, y_pred, gamma=2., alpha=4., epsilon=1.e-9):
    y_true_c = tf.convert_to_tensor(y_true, tf.float32)
    y_pred_c = tf.convert_to_tensor(y_pred, tf.float32)

    model_out = tf.add(y_pred_c, epsilon)
    ce = tf.multiply(y_true_c, -tf.math.log(model_out))
    weight = tf.multiply(y_true_c, tf.pow(tf.subtract(1., model_out), gamma))
    fl = tf.multiply(alpha, tf.multiply(weight, ce))

    reduced_fl = tf.reduce_max(fl, axis=-1)
    return tf.reduce_mean(reduced_fl)


def hybrid_loss(y_true, y_pred):
    j_loss = jaccard_loss(y_true, y_pred)
    d_loss = dice_loss(y_true, y_pred)
    f_loss = focal_loss(y_true, y_pred)
    return j_loss + d_loss + f_loss


class VisualizeProgress(tf.keras.callbacks.Callback):
    def __init__(self, test_data, every_n_epochs=10, num_samples=4,
                 deep_supervision=False, log=False):
        super(VisualizeProgress, self).__init__()
        self.test_data = test_data
        self.every_n_epochs = every_n_epochs
        self.num_samples = num_samples
        self.deep_supervision = deep_supervision
        self.log = log

    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) % self.every_n_epochs == 0:
            x_test, y_test = self.test_data
            y_pred = self.model.predict(x_test[:self.num_samples])

            if self.deep_supervision:
                y_pred = y_pred[-1]

            plt.figure(figsize=(9, 12))
            for index in range(self.num_samples):
                if index >= len(x_test): break

                plt.subplot(self.num_samples, 3, index * 3 + 1)
                if index == 0:
                    plt.title('Input Image')
                plt.imshow(x_test[index])

                plt.subplot(self.num_samples, 3, index * 3 + 2)
                if index == 0:
                    plt.title('Ground Truth')
                ground_truth = one_hot_to_rgb(y_test[index]).numpy()
                plt.imshow(ground_truth)

                plt.subplot(self.num_samples, 3, index * 3 + 3)
                if index == 0:
                    plt.title('Predicted Segmentation')
                prediction = one_hot_to_rgb(y_pred[index]).numpy()
                plt.imshow(prediction)

            if self.log:
                wandb.log({f'Segmentation results': wandb.Image(plt)})

            plt.tight_layout()
            plt.show()


In [ ]:
# SM losses
sm_dice = sm.losses.DiceLoss(class_weights=CLASS_WEIGHTS)
sm_focal = sm.losses.CategoricalFocalLoss()
sm_cce = sm.losses.CategoricalCELoss(class_weights=CLASS_WEIGHTS)
sm_jac = sm.losses.JaccardLoss(class_weights=CLASS_WEIGHTS)
# sm_total_loss = (1. * sm_dice) + (3. * sm_focal) + (1. * sm_cce) + (1. * sm_jac)
sm_total_loss = (1. * sm_dice) + (3. * sm_focal)
# sm_total_loss = sm.losses.DiceLoss() + sm.losses.BinaryFocalLoss()

# SM metrics
sm_iou = sm.metrics.IOUScore(threshold=0.5)
sm_f1 = sm.metrics.FScore(threshold=0.5)

In [ ]:
model = UNet(
    input_shape=IMAGE_SHAPE,
    num_classes=NUM_CLASSES,
    num_filters=32,
    dropout=DROPOUT,
    batchnorm=True,
    final_activation='softmax',
)

model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE, beta_1=MOMENTUM),
    loss='categorical_crossentropy',
    metrics=['accuracy', jaccard_coef, dice_coef, sm_iou, sm_f1],
)

tf.keras.utils.plot_model(model, show_shapes=True)
model.summary()

In [ ]:
# Smaller dataset for testing
EPOCHS = 3
STEPS_PER_EPOCH = 2
train_ds = train_ds.take(STEPS_PER_EPOCH)
val_ds = val_ds.take(STEPS_PER_EPOCH)

callbacks = [
    ReduceLROnPlateau(monitor='val_loss', factor=0.1, patience=5, verbose=1),
    EarlyStopping(monitor='val_loss', patience=10, verbose=1),
    VisualizeProgress(next(iter(val_ds)), every_n_epochs=1, num_samples=5, log=LOGGING),
]
if LOGGING:
    callbacks.append(WandbCallback())

history = model.fit(
    train_ds, validation_data=val_ds, validation_steps=len(val_ds),
    epochs=EPOCHS, steps_per_epoch=STEPS_PER_EPOCH,
    callbacks=callbacks, verbose=1,
)

In [ ]:
# Plot training history
plt.figure(figsize=(12, 8))

plt.subplot(221)
plt.title('Loss')
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='val')
plt.legend()

plt.subplot(222)
plt.title('Accuracy')
plt.plot(history.history['accuracy'], label='train')
plt.plot(history.history['val_accuracy'], label='val')
plt.legend()

plt.subplot(223)
plt.title('Jaccard')
plt.plot(history.history['jaccard_coef'], label='train')
plt.plot(history.history['val_jaccard_coef'], label='val')
plt.legend()

plt.subplot(224)
plt.title('Dice')
plt.plot(history.history['dice_coef'], label='train')
plt.plot(history.history['val_dice_coef'], label='val')
plt.legend()

plt.tight_layout()
plt.savefig('training-history.png')
plt.show()

In [ ]:
# save model
model.save('models/basic-unet-model.h5')

In [ ]:
if LOGGING:
    wandb.finish()

## Evaluation

In [ ]:
# load model
model = tf.keras.models.load_model('models/basic-unet-model.h5', custom_objects={
    'jaccard_coef_loss': lambda y_true, y_pred: -jaccard_coef(y_true, y_pred),
    'jaccard_coef': jaccard_coef,
    'dice_coef': dice_coef,
})

In [ ]:
# load model for testing
model = tf.keras.models.load_model('models/sm-unet-model.h5', compile=False)

In [ ]:
test = next(iter(test_ds))
model.evaluate(test[0], test[1])

In [ ]:
test = next(iter(test_ds))

predictions = model.predict(test[0])
predictions.shape